# `main_96x96` — Full Walkthrough

This notebook explains the **`main_96x96`** training run end-to-end: what was trained, what the model sees, what diffusion actually predicts, how DDIM produces actions at inference, and how the real-time control loop fits inside a 30 Hz budget.

**The run:** `runs/main_96x96/20260528_185334/` — 300 epochs, final train loss `0.00298`, EMA weights.

**The task:** SO-ARM101 6-DOF arm picks one of two red cubes (50/50 left vs right) and drops it in a blue cup. The bimodality is the whole point — this is what a regression baseline cannot represent.

## 1. Configuration — every knob, when it's used

All keys that define this run plus the inference block, with a column for **when** each is actually consulted.

Legend for the **When** column:
- **train** — only read by `train.py`; ignored at inference.
- **infer** — only read by `infer.py`; ignored during training.
- **both** — architectural or shape-defining: a value baked into the model that must match at both ends. Changing it implies retraining.

### Training & model config

| Key | Value | When | Why |
|---|---|---|---|
| `dataset.image_size` | `[96, 96]` | **both** | Train: resizes frames before encoder. Infer: must use same size so encoder sees in-distribution input. |
| `dataset.fps` | `30` | **both** | Train: defines `delta_timestamps` between obs/action samples. Infer: paces `send_action` to match. |
| `dataset.preload_cache` | `true` | train | Decides whether the frame cache lives in RAM. Has no inference equivalent. |
| `dataset.camera_keys` | `[front, wrist]` | **both** | Train: which feature columns to load. Infer: builds the encoder's per-camera ResNet dict. |
| `dataset.state_key` / `action_key` | `observation.state` / `action` | train | Names columns in the LeRobotDataset. Infer reads the state directly off the robot. |
| `model.obs_horizon` (T_o) | `2` | **both** | Train: sample window length. Infer: size of the rolling obs buffer. |
| `model.pred_horizon` (T_p) | `16` | **both** | Train: target action chunk length. Infer: denoiser output shape. Fixed-shape network — can't change without retraining. |
| `model.exec_horizon` (T_a) | `4` | infer | How many chunk steps to actually execute before replanning. Training never uses it. |
| `encoder.backbone` | `resnet18` | **both** | Architecture — must match between training and inference. |
| `encoder.pretrained` | `true` | train | Loads ImageNet weights at module init. At inference the checkpoint already contains the trained weights. |
| `encoder.freeze_bn` | `false` | train | Controls BatchNorm.train() behaviour during training. Eval mode at inference ignores it. |
| `encoder.feature_dim` | `512` | **both** | Architecture. |
| `encoder.state_embed_dim` | `64` | **both** | Architecture. |
| `diffusion.num_train_timesteps` (T) | `100` | **both** | Train: t ~ Uniform{0..T-1}. Infer: DDIM subsamples N steps from this range. |
| `diffusion.noise_schedule` | `cosine` | **both** | Defines the β_t / ᾱ_t arrays used by both noising (train) and DDIM step (infer). |
| `diffusion.beta_start` / `beta_end` | `0.0001` / `0.02` | **both** | Only consulted when `noise_schedule == "linear"`. Unused for cosine. |
| `diffusion.sampler` | `ddpm` (overridden to `ddim` in `infer.py`) | infer | Training loss is sampler-agnostic — picks random t and learns to predict ε. |
| `diffusion.num_inference_steps` | `100` (overridden by `infer.num_ddim_steps`) | infer | Number of DDIM denoiser passes per chunk. |
| `diffusion.clip_sample` | `true` | infer | Clipping happens *inside* the DDIM step on the x̂₀ estimate. Training MSE on ε doesn't clip. |
| `diffusion.prediction_type` | `epsilon` | **both** | Train: loss target is the noise. Infer: the same convention is used to invert ε → x̂₀. |
| `denoiser.backbone` | `cnn` | **both** | Architecture (1-D U-Net vs transformer). |
| `denoiser.cnn.channels` | `[256, 512, 1024]` | **both** | Architecture. |
| `denoiser.cnn.kernel_size` | `5` | **both** | Architecture. |
| `denoiser.cnn.n_groups` | `8` | **both** | GroupNorm config — architecture. |
| `training.seed` | `42` | train | RNG seed for dataset shuffle, weight init, noise sampling. |
| `training.experiment` | `main_96x96` | train | Output sub-directory name under `runs/`. |
| `training.batch_size` | `256` | train | Mini-batch size. Inference always runs B=1. |
| `training.num_workers` | `6` | train | DataLoader workers. |
| `training.num_epochs` | `300` | train | Length of the training schedule. |
| `training.lr` | `1e-4` | train | AdamW learning rate. |
| `training.weight_decay` | `1e-6` | train | AdamW weight decay. |
| `training.lr_scheduler` | `cosine` | train | LR schedule shape. |
| `training.warmup_steps` | `500` | train | Linear warm-up before cosine decay. |
| `training.ema_decay` | `0.9999` | train | Decay factor applied each gradient step to update the EMA shadow weights. At inference we just load the resulting weights — the decay value itself is no longer used. |
| `training.grad_clip` | `1.0` | train | Global grad-norm clip. |
| `training.log_every` | `100` | train | TensorBoard log cadence. |
| `training.run_dir` | `runs` | train | Output base directory. |
| `training.use_amp` | `true` | train | bf16/fp16 autocast for the training forward+backward. Inference uses plain fp32 (or whatever `torch.compile` chooses). |

### Inference-only config (`infer:` block)

| Key | Value | When | Why |
|---|---|---|---|
| `infer.robot_port` | `/dev/serial/by-id/...USB_Single_Serial...` | infer | USB serial device for the Feetech motor bus. |
| `infer.robot_id` | `valera` | infer | Filename stem under `~/.cache/huggingface/lerobot/calibration/robots/so_follower/` — selects which calibration to load. |
| `infer.cameras.{front,wrist}.path` | `/dev/v4l/by-id/...` | infer | V4L2 device path per camera. |
| `infer.cameras.{front,wrist}.width` / `height` | `640` / `480` | infer | Native capture resolution; resized to `dataset.image_size` before the encoder. |
| `infer.cameras.{front,wrist}.fps` | `30` | infer | Camera sensor fps. Must match `dataset.fps` so the implicit velocity (state at t vs t−1) matches training. |
| `infer.cameras.{front,wrist}.fourcc` | `MJPG` | infer | Motion-JPEG over USB — compressed stream, reduces bandwidth vs raw YUY2. |
| `infer.cameras.{front,wrist}.backend` | `200` | infer | `cv2.CAP_V4L2` — forces the V4L2 kernel driver, prevents OpenCV falling back to GStreamer (which adds pipeline latency). |
| `infer.camera_key_map` | `{front: observation.images.front, wrist: observation.images.wrist}` | infer | Maps robot's short camera name → the dataset feature key the encoder expects. |
| `infer.num_ddim_steps` | `10` | infer | Overrides `diffusion.num_inference_steps`. Number of DDIM denoiser passes per chunk. |
| `infer.compile` | `true` | infer | Whether to wrap the model with `torch.compile`. ~30 s warmup, then ~25-30% faster steady-state. |

In [1]:
# Inspect the actual saved config for this run
from pathlib import Path
from omegaconf import OmegaConf

RUN_DIR = Path("../../runs/main_96x96/20260528_185334")
cfg = OmegaConf.load(RUN_DIR / "config.yaml")
print(OmegaConf.to_yaml(cfg))

dataset:
  path: recordings/redcubes_bluecup
  camera_keys:
  - observation.images.front
  - observation.images.wrist
  state_key: observation.state
  action_key: action
  fps: 30
  image_size:
  - 96
  - 96
  preload_cache: true
model:
  obs_horizon: 2
  pred_horizon: 16
  exec_horizon: 4
encoder:
  backbone: resnet18
  pretrained: true
  freeze_bn: false
  feature_dim: 512
  state_embed_dim: 64
diffusion:
  num_train_timesteps: 100
  noise_schedule: cosine
  beta_start: 0.0001
  beta_end: 0.02
  sampler: ddpm
  num_inference_steps: 100
  clip_sample: true
  prediction_type: epsilon
denoiser:
  backbone: cnn
  cnn:
    channels:
    - 256
    - 512
    - 1024
    kernel_size: 5
    n_groups: 8
  transformer:
    n_heads: 8
    n_layers: 6
    d_model: 256
    dropout: 0.1
training:
  seed: 42
  experiment: main_96x96
  batch_size: 256
  num_workers: 6
  num_epochs: 300
  lr: 0.0001
  weight_decay: 1.0e-06
  lr_scheduler: cosine
  warmup_steps: 500
  ema_decay: 0.9999
  grad_clip: 1.0


## 2. What the model sees and what it predicts

Per training sample (`obs_horizon=2`, `pred_horizon=16`):

```
INPUT (observation window, 2 timesteps):
  images = {
    "observation.images.front":  (2, 3, 96, 96)   # two RGB frames, normalised to [0,1]
    "observation.images.wrist":  (2, 3, 96, 96)   # wrist camera, same shape
  }
  state = (2, 6)                                  # joint angles in degrees at t-1, t

OUTPUT (action chunk, 16 future timesteps):
  actions = (16, 6)                               # target joint angles in degrees
```

**You asked: "by one picture of wrist and front camera plus given position"** — almost. The model sees **two** consecutive frames per camera (T_o=2), not one. The second frame is "now", the first is one tick earlier (33 ms ago at 30 Hz). Same for the state vector. This is what gives the model implicit access to velocity without needing to compute derivatives.

**The action chunk is 16 steps long** (`pred_horizon=16`). That's a 16-step plan into the future (≈ 533 ms at 30 Hz). In the saved run we only execute the first `exec_horizon=4` steps before replanning.

**The 6 dimensions** correspond to the 6 SO-ARM101 motors, in this fixed order:
```
['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
```

### The `actions` variable — structure in detail

What `model.predict_actions(batch)` returns:

```
actions:    torch.Tensor of shape (B=1, T_p=16, action_dim=6)
dtype:      float32
device:     cuda  (we move to cpu before sending to the arm)
units:      degrees  (already denormalised — passed through action_normalizer.inverse)
semantic:   ABSOLUTE target joint angles, NOT velocities and NOT deltas
time axis:  actions[0, i, :] is the target pose at t + (i+1) ticks, ticks 33.3 ms apart
joint axis: same fixed motor order as listed above
```

Concretely, when `_run_loop` sends one step:

```python
action_vec = actions[0, i].cpu().numpy()                       # shape (6,)
action_dict = {f"{m}.pos": float(v) for m, v in zip(motor_names, action_vec)}
robot.send_action(action_dict)
```

the dict actually sent over USB looks like:

```python
{
  'shoulder_pan.pos':   -8.13,
  'shoulder_lift.pos': -103.56,
  'elbow_flex.pos':     96.88,
  'wrist_flex.pos':     52.99,
  'wrist_roll.pos':      4.78,
  'gripper.pos':         4.59,
}
```

Each value is the **absolute desired motor angle in degrees** at that future tick. We do not send velocities, torques, accelerations, or relative deltas — we send positions, and the motor's onboard controller decides how to get there (see Section 9).

## 3. Architecture — the two networks

There are exactly two trainable networks: the **observation encoder** and the **denoiser**.

### 3a. ObservationEncoder (~22M params)

Per camera, per timestep:

```
image (3, 96, 96) in [0,1]
  → ImageNet normalisation: (x - mean) / std
  → ResNet18 (no FC head, just up to global avg pool) → (512,)
  → Linear(512 → 512)                                  → (512,) feature vector
```

Two cameras × two timesteps × 512-d = **2048 image features**. Plus state:

```
state (6,) per timestep → Linear(6 → 64) + SiLU → (64,)
```

Two timesteps × 64-d = **128 state features**.

**Concatenated conditioning vector:** `(2048 + 128) = 2176-d`. This is the entire context the denoiser ever sees about the world.

Notes:
- Each camera has its **own ResNet18** (no weight sharing across cameras — they observe different statistics).
- ResNet18 weights **are shared across the 2 obs_horizon frames** of the same camera (temporal weight sharing).
- The encoder is **trained end-to-end** with the denoiser — the ImageNet weights are just a warm start.

### 3b. Denoiser — 1-D temporal U-Net with FiLM (~47M params)

Inputs to one forward pass: `x_t ∈ ℝ^(T_p=16, action_dim=6)`, `t ∈ {0,…,99}`, `cond ∈ ℝ^2176`.

Structure:

```
Conditioning preparation (done once per forward):
  t  ─sinusoid──→ (256,) ──MLP──→ (512,)──┐
  cond (2176,) ─Linear─→ (512,)──────────┤
                                          ▼
                                       cond_emb (B, 512)
                                       used by every FiLM block

Encoder path on x_t (treating T_p=16 as the "time" axis of a 1-D conv):
  Level 0:  Conv1d(6 → 256)  ┌─ 2 × FiLMResBlock(256, cond_emb)   → skip₀ at (B, 256, 16)
                              └─ AvgPool stride 2                  → (B, 256, 8)
  Level 1:  Conv1d(256 → 512) ┌─ 2 × FiLMResBlock(512, cond_emb)  → skip₁ at (B, 512, 8)
                              └─ AvgPool stride 2                  → (B, 512, 4)
  Middle:   2 × FiLMResBlock(1024, cond_emb) at (B, 1024, 4)

Decoder path (mirrors encoder, with skip concat):
  Up → cat(skip₁) → 2 × FiLMResBlock(1024→512, cond_emb)
  Up → cat(skip₀) → 2 × FiLMResBlock(512→256, cond_emb)
  Conv1d(256 → 6)  → noise prediction ε̂ of shape (B, 16, 6)
```

**FiLM (Feature-wise Linear Modulation)** is how the conditioning vector affects the denoiser. Inside each residual block:

```
(γ, β) = Linear(cond_emb)                       # both ∈ ℝ^out_channels
h      = Conv1d(input)
h      = GroupNorm(h)
h      = (1 + γ) · h + β                        # per-channel scale & shift
h      = SiLU(h)
```

This is the only path conditioning takes into the denoiser — there is **no attention, no cross-attention, no classifier-free guidance**. The whole observation vector becomes per-channel γ and β scaling factors injected at every U-Net block.

In [ ]:
# Inspect parameter counts and verify shapes
import sys
sys.path.insert(0, "../..")

import torch
from omegaconf import OmegaConf
from diffusion_policy_soarm.models.diffusion import DiffusionModule, build_denoiser
from diffusion_policy_soarm.models.encoders import ObservationEncoder
from diffusion_policy_soarm.data.normalization import Normalizer

cfg = OmegaConf.load("../../runs/main_96x96/20260528_185334/config.yaml")

anorm = Normalizer.from_file("../../runs/main_96x96/20260528_185334/action_normalizer.json")
snorm = Normalizer.from_file("../../runs/main_96x96/20260528_185334/state_normalizer.json")
action_dim, state_dim = anorm.mins.shape[0], snorm.mins.shape[0]

enc = ObservationEncoder(cfg, camera_keys=list(cfg.dataset.camera_keys), state_dim=state_dim)
den = build_denoiser(cfg, action_dim=action_dim, obs_cond_dim=enc.output_dim)

def count(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f"Encoder:  {count(enc):.1f}M params, output dim = {enc.output_dim}")
print(f"Denoiser: {count(den):.1f}M params")
print(f"Total:    {count(enc) + count(den):.1f}M params")

# Sanity-check shapes with a dummy batch
B = 1
imgs = {k: torch.zeros(B, cfg.model.obs_horizon, 3, *cfg.dataset.image_size) for k in cfg.dataset.camera_keys}
state = torch.zeros(B, cfg.model.obs_horizon, state_dim)
cond = enc(imgs, state)
print(f"\nCond vector shape: {cond.shape}  (= {cfg.model.obs_horizon} × (2 × {cfg.encoder.feature_dim} + {cfg.encoder.state_embed_dim}) = {enc.output_dim})")

xt = torch.zeros(B, cfg.model.pred_horizon, action_dim)
t = torch.zeros(B, dtype=torch.long)
eps = den(xt, t, cond)
print(f"Predicted noise shape: {eps.shape}  (= batch × pred_horizon × action_dim)")

## 4. Diffusion — training math

### 4a. What we're modelling

The data distribution is `p(actions | observations)` — the conditional distribution of 16-step action chunks given the observation window. Because picking a *left* cube and picking a *right* cube are equally good, this distribution is **bimodal**. A regression model would average them and pick neither.

Diffusion sidesteps the averaging problem by learning the **score** (gradient of log-density), then sampling from it.

### 4b. The forward (noising) process

Take a clean normalised action chunk `x_0 ∈ ℝ^(16, 6)`. Define a noise schedule `β_t` for `t = 0…99` (we use cosine). Let:

```
α_t  = 1 − β_t
ᾱ_t  = ∏_{s≤t} α_s
```

Then for any timestep `t`:

```
x_t = √ᾱ_t · x_0 + √(1 − ᾱ_t) · ε,    ε ~ N(0, I)
```

At `t=0`, `x_t ≈ x_0` (clean). At `t=T−1=99`, `x_t ≈ ε` (pure noise). Cosine schedule keeps signal alive longer than linear → easier for the network at large t.

### 4c. Training loss

One training step:

1. Sample a batch of `(observation, x_0)` from the dataset.
2. Sample `t ~ Uniform{0, …, 99}` per example.
3. Sample `ε ~ N(0, I)` with the shape of `x_0`.
4. Compute `x_t` from the equation above.
5. Get `cond = encoder(observation)`.
6. Predict `ε̂ = denoiser(x_t, t, cond)`.
7. Loss: `MSE(ε̂, ε)`.

**That's it.** The model learns to predict the noise that was added to a clean action chunk, conditional on what the cameras and proprio saw. There is no separate "action loss" — predicting noise is mathematically equivalent (up to constants) to learning the score `∇_x log p(x|c)`.

## 5. DDIM inference — what actually happens when you run `infer.py`

### 5a. DDPM (full ancestral) vs DDIM (deterministic, sub-sampled)

DDPM at inference: walk back **all 100 training steps**, adding fresh noise at each step. Slow but stochastic and gold-standard.

DDIM at inference: pick a **sub-sample** of N steps from the 100 training steps (we use `N = 10` typically), and take a *deterministic* (η=0) reverse step at each. ~10× fewer denoiser calls, same quality.

### 5b. The DDIM step in our code

For each chosen timestep `t` (going from large → small):

```
ε̂ = denoiser(x_t, t, cond)                           # one network forward

# Reconstruct estimate of the clean action chunk
x̂_0 = (x_t − √(1−ᾱ_t) · ε̂) / √ᾱ_t

# Clip — this is where shoulder_lift and elbow_flex got pinned in your run
x̂_0 = clip(x̂_0, −1, +1)            # because clip_sample: true

# Step to the next (smaller) timestep deterministically
x_{t_prev} = √ᾱ_{t_prev} · x̂_0 + √(1 − ᾱ_{t_prev}) · ε̂
```

After all N steps, `x_t ≈ x_0`. We then **denormalise** through the Normalizer to convert from [−1, 1] back into degrees.

### 5c. Number of denoiser forwards per inference

**`num_ddim_steps` forwards through the 47M-param U-Net per action chunk.**

- 5 steps  → ~26 ms inference (fast, slight quality loss)
- 10 steps → ~50 ms (paper-recommended sweet spot)
- 15 steps → ~75 ms
- 100 steps → ~530 ms (= same speed as full DDPM; serial bus times out!)

**The encoder runs only once per inference** — its output `cond` is reused across all N denoiser steps.

## 6. Inference end-to-end (`infer.py` walkthrough)

Numbered with the source-code lines as of writing:

```
[once at startup]
  1. Load YAML config + apply CLI overrides.
  2. Override sampler → DDIM, num_inference_steps → cfg.infer.num_ddim_steps.
  3. Build encoder + denoiser, instantiate DiffusionModule.
  4. Load EMA weights from checkpoint["ema"] (NOT the raw `model` key).
  5. Optionally torch.compile(model) — ~30 s warmup, then steady-state.
  6. Build SOFollower with id='valera' so the calibration JSON loads from
     ~/.cache/huggingface/lerobot/calibration/robots/so_follower/valera.json.
  7. robot.connect() — bus.is_calibrated is already True, no prompt.
  8. Pre-fill a deque(maxlen=obs_horizon=2) with two initial observations.

[main loop, until Ctrl-C]
  9. obs = robot.get_observation()              # serial round-trip + camera grabs
 10. obs_buffer.append(obs)
 11. Build the batch dict:
     - For each camera: stack the two frames, permute to (T_o, C, H, W),
       float / 255, bilinear-resize 480x640 → 96x96, add batch dim.
     - For state: stack two motor-position dicts in the fixed motor order.
 12. model.predict_actions(batch):
       state_norm = state_normalizer(state)
       cond = encoder(images, state_norm)        # 1 encoder pass
       x = N(0, I) of shape (1, 16, 6)
       for t in N timesteps: x = DDIM_step(...)  # N denoiser passes
       actions = action_normalizer.inverse(x)    # → degrees
 13. Print latency stats.
 14. For i in 0..exec_horizon-1 (=0..7 by default):
       action_dict = {f"{motor}.pos": actions[0, i, j] for j, motor in enumerate(motor_names)}
       robot.send_action(action_dict)            # serial write
       time.sleep(1/fps - send_overhead)         # pace to 30 Hz
 15. Loop back to 9.
```

**Receding horizon, in one sentence:** predict 16 future action steps, execute 8, then re-observe and predict 16 fresh ones. We don't blindly execute all 16 because the world changes.

## 7. The three horizons — `obs_horizon`, `pred_horizon`, `exec_horizon`

These are the most confusing knobs in the whole policy. Concrete picture:

```
Real-world wall clock at 30 Hz, each tick = 33.3 ms.

        past               now                      future
   ...─┬───┬───┬───┬───┬───●───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬─...
       t-1  t                t+1 t+2 ...
       │═══════│                │═════════════════════════════════│
       │ T_o=2 │                │            T_p = 16             │
       observation             predicted action chunk
       window
                                │═════════════════│
                                │     T_a = 4     │
                                actually executed
```

- **`obs_horizon = T_o`**: how many past+current frames feed the encoder. We use 2 → the model implicitly sees one-step velocity.
- **`pred_horizon = T_p`**: how many future action steps the denoiser outputs in one shot. We use 16 → ~533 ms of plan. The denoiser is a fixed-shape network: change this, retrain.
- **`exec_horizon = T_a`**: how many of those 16 we execute before throwing the rest away and replanning. We use 8 → re-observe every 267 ms.

**`exec_horizon ≤ pred_horizon`** always (we can't execute steps that weren't predicted). The bigger `exec_horizon`, the longer we trust the open-loop plan.

## 8. FPS, the 30 Hz budget, and what slowdowns cause

### 8a. What "30 fps" actually means in our control loop

The arm needs an action commanded every **33.3 ms** to stay smooth. The control loop sends one motor command per tick (line 14 in the walkthrough). Between two replans we only send commands — no inference — so 33 ms/step is easy.

The hard budget is the **inference burst** that happens once per `exec_horizon` ticks. We sit idle for `exec_horizon × 33 ms` and then need to produce 16 new actions before the next tick. If inference is faster than the gap between two replans, we're fine; if it's slower, the arm jitters or the serial bus disconnects.

Concretely with `exec_horizon = 4`:
```
max acceptable inference time ≈ 8 × 33 ms = 267 ms
                                            ^ above this, the next send_action is late
```
But in practice the serial bus times out earlier (~500 ms with default Feetech settings) — that's the **"no status packet"** error you hit at 100 DDIM steps.

### 8b. What happens if inference is slower than 30 fps per single step?

Two failure modes, in order of severity:

1. **Soft (jitter, lag).** If one inference call exceeds the budget, the next batch of `exec_horizon` action steps starts late. The robot looks twitchy because each replan's actions don't smoothly continue the previous chunk — the model sees slightly stale observations. The arm doesn't fall apart, it just wobbles. **This is what you saw with the 5-step run.**

2. **Hard (serial timeout).** If a single inference call takes >~500 ms, the USB serial bus to the Feetech motors hits its packet timeout and drops the connection. You get `[TxRxResult] There is no status packet!` and the script crashes. **This is what you saw with 100 DDIM steps.**

### 8c. So can we just lower the camera fps?

No — and here's why this is a really common confusion:

The cameras run at **30 fps natively** and the model was *trained* on 30 fps data, so the 33-ms gap between frames is baked into what the model expects to see between consecutive observations. If you slow the cameras down to 15 fps, the model sees observation gaps that are 2× larger than during training — its implicit velocity estimate (state at t vs t−1) is wrong, and behaviour degrades.

**Camera fps is set by the data distribution at training time and you can't change it at inference.** Same goes for the control rate.

### 8d. What you actually have to tune

If inference is too slow, the right knobs are:

- **`infer.num_ddim_steps`** — fewer denoiser passes per chunk. 5–15 is the practical range. Below 5 you can see quality drop; above 20 there's no quality gain.
- **`model.exec_horizon`** — execute more steps per replan → fewer replans per second → easier budget. But longer open-loop → less reactive to disturbances.
- **`infer.compile`** — `torch.compile` (~30 s warmup) gives ~30% speedup steady-state.
- **`dataset.image_size`** — smaller images → faster encoder. But this changes training, not just inference; you'd need a separate run.

## 9. Motor control — how `send_action` actually moves the arm, and why inference feels faster than teleop

This section answers a real question that came up on this run: **"the arm moved way too fast and scary at inference, faster than when I was teleoperating it during recording. Why?"**

### 9a. Is there a PID controller? Where does it live?

**Yes — but we don't implement it.** Every Feetech STS3215 servo on the SO-ARM101 has a **PID position controller burned into the motor's firmware**. When you write a `Goal_Position` register over the serial bus, the servo's microcontroller runs its own closed-loop PID at >1 kHz to drive the shaft to that target. There is no PID code on our side — none in `diffusion_policy_soarm`, none in `lerobot`, none in `scservo_sdk`.

What lives on each motor (firmware-side):

| Register | What it does | Where it's set |
|---|---|---|
| `Goal_Position` | Target shaft angle (raw ticks) | `robot.send_action(...)` writes this every tick |
| `Goal_Velocity` / `Goal_Time` | (Optional) speed cap or trajectory duration | Left at factory defaults |
| `Acceleration` | Trajectory-generator ramp rate | Set to **254** (≈ max) by `SOFollower.configure_motors()` on connect |
| `Maximum_Acceleration` | Hard ceiling on acceleration | Set to **254** (≈ max) by `SOFollower.configure_motors()` on connect |
| PID gains (Kp, Ki, Kd) | Closed-loop tuning inside the motor | Factory defaults, not touched by LeRobot |
| `Min_Position_Limit` / `Max_Position_Limit` | Safety bounds in raw ticks | Written from the calibration JSON on first connect |

So the full control stack at inference:

```
diffusion policy             writes:  Goal_Position (degrees → ticks)         every 33 ms
SOFollower.send_action       ──────────────────────────────────────────►
scservo_sdk over USB                                                            (~2 ms)
Feetech STS3215 firmware     runs:   internal PID @ >1 kHz to chase Goal_Position
                                      respecting Acceleration / Max_Accel limits
mechanical shaft             moves toward Goal_Position
```

We send *positions* and the motor does the dirty work. That keeps our Python loop simple — but it also means **the speed at which the arm moves is controlled by the motor firmware, not by our code**.

### 9b. Why was inference scary-fast vs teleop?

The model is correct, the policy is correct, the motors are correct — and yet the arm slams to each new target. Three contributing causes:

1. **Big inter-step position deltas in the predicted chunk.** During teleop, the leader arm moves at human hand speed (~10-30 deg/s). Each tick the goal moves ~1 deg — the motor barely needs to accelerate. At inference, the diffusion model can predict targets that jump 5–20 degrees between consecutive steps of the chunk, because the training data contains some fast motions and the denoiser interpolates between them. When the motor sees a 20-degree delta in 33 ms, it accelerates at its configured max (254/255) and chases the target at full speed.

2. **`SOFollower.configure_motors(acceleration=254, maximum_acceleration=254)` is called on every `robot.connect()`.** This is by design (default is `254` ≈ max in `lerobot/robots/so_follower/so_follower.py`). It makes teleop responsive but turns each commanded delta into a fast move.

3. **`exec_horizon` makes it worse.** If you bumped `exec_horizon` to 16 to fight jitter, the arm now gets 16 waypoints in a row to chase. Each one is an absolute position, and the motor slams to each as fast as its acceleration limit allows. Larger `exec_horizon` = longer uninterrupted aggressive motion before the next replan.

### 9c. Fixes, ordered from quickest to most thorough

**1. Keep `exec_horizon` small.** The saved run uses `exec_horizon=4`, which limits how many absolute targets the arm chases before the next replan.

**2. Use warm-started replanning.** The current `infer.py` shifts the unexecuted suffix of the previous chunk into the next DDIM initialisation instead of restarting from fresh Gaussian noise every time. This reduces chunk-boundary discontinuities.

**3. Lower the motor's configured `Acceleration` register.** A one-time write (`robot.bus.write("Acceleration", motor, 64)` or similar) caps how aggressively the firmware ramps. Persistent until power cycle. Affects teleop too, so consider this a deployment-time tweak.

**4. Verify the predicted action is in the right units.** We already confirmed this — the printed values (~−100 to ~100 deg) sit inside the normalizer's degree range, so denormalisation is one-pass and correct. Worth a sanity check if you ever change normalizer code.

**5. Re-record training data with a speed cap on the leader arm.** Long term: if you want the policy itself to learn slower trajectories, train it on data that was slower. The policy reproduces the distribution of speeds it saw.

### 9d. Why don't we use a software PID on top?

We could — wrap `robot.send_action` so that each commanded position is the output of a Python-side PID with the diffusion policy's prediction as the setpoint and the current motor position as the measurement. But:

- The motor's onboard PID runs at >1 kHz with no jitter; a Python-side PID would run at our 30 Hz control rate. Adding a slower outer loop on top of a faster inner loop usually makes things worse, not better.
- The right knob for "go slower" is acceleration / per-step delta limits, not PID gains. PID controls how aggressively to *converge*, not how fast to *travel*.

For this repo's current control loop, warm-started replanning plus the motor's own acceleration limit are the intended deployment controls.

## 10. CPU vs GPU — where the work happens

Per inference (one call to `model.predict_actions`):

| Stage | Device | Notes |
|---|---|---|
| Camera frame grab | CPU (USB) | OpenCV V4L2 → numpy uint8 |
| Serial read (motor positions) | CPU (USB) | Feetech SCS protocol via scservo_sdk |
| Image preprocessing (permute, float, /255, bilinear resize) | CPU then transfer | 480×640 → 96×96; tiny work, mostly memcpy |
| `state_normalizer(state)` | GPU | tensor op |
| `encoder(images, state)` | GPU | 2 cameras × 2 frames = 4 ResNet18 forwards |
| DDIM loop (N denoiser forwards) | GPU | The bulk of inference time |
| `action_normalizer.inverse(x)` | GPU | trivial |
| `actions.cpu().numpy()` | GPU → CPU sync | one tiny transfer |
| Serial write (motor commands) | CPU (USB) | per exec_horizon step |

**GPU dominates the wall-clock time** when DDIM steps are high; otherwise serial latency starts to compete.

There is no GPU↔CPU ping-pong during the DDIM loop — `cond` stays on GPU, `x_t` stays on GPU, only the final action tensor crosses back at the end.

### Is `torch.compile` worth it?

Yes — but it's a one-time ~30 s warmup at startup as Inductor traces the graph. After that, the same N denoiser calls are noticeably faster (~25-30% on our 47M U-Net). Set `infer.compile: false` if you're iterating quickly on infer.py and don't want to pay the warmup.

## 11. How to improve inference latency further

Ranked by effort vs payoff:

1. **Drop DDIM steps to 5–10.** Free. First thing to try.
2. **Keep `torch.compile` on.** Free after the 30 s warmup.
3. **Bump `exec_horizon` to 12 or 16.** Halves the replan rate. Costs you closed-loop reactivity.
4. **Run inference in a background thread**, control loop always sends from the most recent completed chunk at 30 Hz. Decouples latency from control entirely. ~50 lines of code; worth it if you can't hit budget any other way.
5. **Smaller image size** (e.g. 64×64) — requires retraining.
6. **Quantise / fp16 the denoiser** — `torch.compile` + AMP already does this implicitly; explicit int8 would require a calibration pass.
7. **Distill DDIM into 1-step consistency model** — paper-grade work, weeks of effort, not needed here.

## 12. Debug recipes

### 12a. Verify normalisation

Pinned-to-bound actions (the `-103.56`/`96.88` you saw) usually mean either (a) the model is correct and the arm is genuinely at an extreme pose, or (b) you're running far from the training distribution and the denoiser saturates `clip_sample`. Check the actual range:

In [ ]:
import json
for name in ["action", "state"]:
    p = json.load(open(f"../../runs/main_96x96/20260528_185334/{name}_normalizer.json"))
    print(f"{name}:")
    for j, (lo, hi) in enumerate(zip(p["mins"], p["maxs"])):
        print(f"  joint {j}: [{lo:8.2f}, {hi:8.2f}]  (range {hi-lo:.2f})")

### 12b. Verify motor ordering matches the dataset

The state vector in the dataset is recorded in a specific motor order. The robot's `bus.motors.keys()` must match it at inference, otherwise the model sees shuffled joint values and gives garbage actions.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset("redcubes_bluecup", root="../../recordings/redcubes_bluecup", download_videos=False)
feat = ds.features["observation.state"]
print("Dataset state column order:", feat.get("names", "(unnamed)"))
print("Robot motor order (hardcoded in SOFollower):")
print("  ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']")

### 12c. Save a camera frame and confirm it looks like the training set

Wrong camera, wrong colour space, or flipped axes are common silent failures. Save one frame from each camera and eyeball it against `recordings/redcubes_bluecup/`.

### 12d. Inspect the predicted actions vs the current state

Uncomment the `print("actions:", ...)` lines in `_run_loop` in `infer.py`. If predicted actions ≈ current state every iteration, the policy thinks "stay still" — usually because the scene doesn't match training (e.g. arm not in starting pose, cube not in frame).

### 12e. Check training loss

```
uv run tensorboard --logdir runs/main_96x96
```

For this run, final loss was `0.00298` over 300 epochs with a smooth cosine decay — model is well-converged. If loss plateaued early or stayed above ~0.01, the model didn't learn the task and no amount of inference tuning will fix it.

## 13. One-paragraph summary you can recite

> `main_96x96` is a Diffusion Policy trained on 100 SO-ARM101 episodes of a bimodal pick-and-place task. Per inference call, the observation encoder runs ResNet18 once over each of two consecutive 96×96 frames from the front and wrist cameras (plus an embedding of the two corresponding 6-D joint states) to produce a 2176-d conditioning vector. A 1-D temporal U-Net with FiLM conditioning then runs DDIM denoising — 10 steps subsampled from the 100-step cosine training schedule — turning Gaussian noise into a 16-step normalised action chunk. The chunk is denormalised back into degrees and sent to the arm as **absolute joint-angle setpoints**; the motors' onboard PID controllers do the actual position control. In the saved run we execute the first 4 of the 16 steps at 30 Hz before re-observing and replanning. EMA weights are used at inference. Total ~70M parameters, ~50 ms per inference on a single RTX-class GPU with `torch.compile`. The bimodality is what motivates diffusion over BC: a Gaussian regressor would average the "go left" and "go right" modes and pick neither, while the denoiser samples one mode per call.